**Information** about data

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("csdepartmentfood/central-asian-food-dataset")

print("Path to dataset files:", path)

100%|██████████| 963M/963M [00:11<00:00, 90.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/csdepartmentfood/central-asian-food-dataset/versions/1


In [2]:
import os
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image
import pandas as pd
import numpy as np

# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = Path(path)

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp"
}

SPLITS = ["train", "val", "test"]

print("=" * 70)
print("CENTRAL ASIAN FOOD DATASET — INTEGRITY AUDIT")
print("=" * 70)

print("Dataset path:")
print(DATASET_PATH)

CENTRAL ASIAN FOOD DATASET — INTEGRITY AUDIT
Dataset path:
/root/.cache/kagglehub/datasets/csdepartmentfood/central-asian-food-dataset/versions/1


In [3]:
# ============================================================
# COLLECT ALL IMAGE FILES
# ============================================================

records = []

for split in SPLITS:

    split_path = DATASET_PATH / split

    if not split_path.exists():
        print(f"[WARNING] Missing split: {split}")
        continue

    for class_dir in sorted(split_path.iterdir()):

        if not class_dir.is_dir():
            continue

        class_name = class_dir.name

        for image_path in class_dir.rglob("*"):

            if (
                image_path.is_file()
                and image_path.suffix.lower() in IMAGE_EXTENSIONS
            ):

                records.append({
                    "split": split,
                    "class": class_name,
                    "filename": image_path.name,
                    "path": str(image_path)
                })

df = pd.DataFrame(records)

print("\nTotal images:", len(df))

print("\nImages by split:")
print(df["split"].value_counts())

print("\nClasses:", df["class"].nunique())

print("\nImages by class:")
print(
    df.groupby(["split", "class"])
      .size()
      .unstack(fill_value=0)
)


Total images: 16402

Images by split:
split
train    10969
val       2735
test      2698
Name: count, dtype: int64

Classes: 42

Images by class:
class  achichuk  airan-katyk  asip  bauyrsak  beshbarmak-w-kazy  \
split                                                             
test         41           46    37        62                 44   
train       205          176    63       300                198   
val          49           56    34        94                 57   

class  beshbarmak-wo-kazy  chak-chak  cheburek  doner-lavash  doner-nan  ...  \
split                                                                    ...   
test                   61         93        94            20         22  ...   
train                 338        456       482            84        104  ...   
val                    70         81        96            35         18  ...   

class  shelpek  shorpa  soup-plain  sushki  suzbe  taba-nan  talkan-zhent  \
split                                  

In [4]:
# ============================================================
# CHECK FOR CORRUPTED / UNREADABLE IMAGES
# ============================================================

corrupted_images = []

for i, row in df.iterrows():

    image_path = Path(row["path"])

    try:
        with Image.open(image_path) as img:
            img.verify()

    except Exception as e:

        corrupted_images.append({
            "split": row["split"],
            "class": row["class"],
            "path": str(image_path),
            "error": str(e)
        })

print("=" * 70)
print("CORRUPTED IMAGE CHECK")
print("=" * 70)

print("Corrupted images:", len(corrupted_images))

if corrupted_images:
    corrupted_df = pd.DataFrame(corrupted_images)
    display(corrupted_df)

else:
    print("PASS — No corrupted images detected.")

CORRUPTED IMAGE CHECK
Corrupted images: 0
PASS — No corrupted images detected.


In [5]:
# ============================================================
# IMAGE DIMENSION ANALYSIS
# ============================================================

dimension_records = []

for i, row in df.iterrows():

    image_path = Path(row["path"])

    try:

        with Image.open(image_path) as img:

            width, height = img.size
            mode = img.mode

            dimension_records.append({
                "split": row["split"],
                "class": row["class"],
                "filename": row["filename"],
                "path": str(image_path),
                "width": width,
                "height": height,
                "mode": mode,
                "aspect_ratio": round(width / height, 3)
            })

    except:
        pass

dimensions_df = pd.DataFrame(dimension_records)

print("=" * 70)
print("IMAGE DIMENSION ANALYSIS")
print("=" * 70)

print("\nUnique dimensions:")
print(dimensions_df[["width", "height"]].drop_duplicates().shape[0])

print("\nMost common dimensions:")
print(
    dimensions_df
    .groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print("\nColor modes:")
print(dimensions_df["mode"].value_counts())

print("\nAspect ratio statistics:")
print(
    dimensions_df["aspect_ratio"].describe()
)

IMAGE DIMENSION ANALYSIS

Unique dimensions:
15223

Most common dimensions:
width  height
640    640       199
1280   720        76
206    206        52
1200   630        28
1502   1064       18
416    416        14
1080   1080       13
1200   900        11
259    194        10
960    720        10
800    800        10
680    482         8
206    196         8
398    498         7
1200   800         7
600    450         6
640    480         6
750    750         6
206    194         6
579    640         6
dtype: int64

Color modes:
mode
RGB    16402
Name: count, dtype: int64

Aspect ratio statistics:
count    16402.000000
mean         1.336130
std          0.490782
min          0.124000
25%          1.016000
50%          1.295000
75%          1.569000
max          7.750000
Name: aspect_ratio, dtype: float64


In [6]:
# ============================================================
# FIND VERY SMALL IMAGES
# ============================================================

small_images = dimensions_df[
    (dimensions_df["width"] < 100) |
    (dimensions_df["height"] < 100)
]

print("Images smaller than 100x100:", len(small_images))

if len(small_images) > 0:
    display(small_images.head(50))

Images smaller than 100x100: 725


,split,class,filename,path,width,height,mode,aspect_ratio
7,train,achichuk,2177.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,126,83,RGB,1.518
39,train,achichuk,4520.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,214,96,RGB,2.229
146,train,achichuk,8257.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,116,87,RGB,1.333
210,train,airan-katyk,5564.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,70,141,RGB,0.496
212,train,airan-katyk,3051.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,88,145,RGB,0.607
213,train,airan-katyk,1101.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,98,230,RGB,0.426
216,train,airan-katyk,3075.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,88,271,RGB,0.325
217,train,airan-katyk,10604.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,87,188,RGB,0.463
237,train,airan-katyk,575.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,122,90,RGB,1.356
254,train,airan-katyk,10853.jpg,/root/.cache/kagglehub/datasets/csdepartmentfo...,84,321,RGB,0.262


In [7]:
import hashlib

# ============================================================
# EXACT FILE DUPLICATE DETECTION
# ============================================================

def calculate_file_hash(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


hash_records = []

for i, row in df.iterrows():

    image_path = Path(row["path"])

    try:

        file_hash = calculate_file_hash(image_path)

        hash_records.append({
            "split": row["split"],
            "class": row["class"],
            "filename": row["filename"],
            "path": str(image_path),
            "file_hash": file_hash
        })

    except Exception as e:

        print("Hash error:", image_path, e)


hash_df = pd.DataFrame(hash_records)

duplicate_groups = (
    hash_df
    .groupby("file_hash")
    .filter(lambda x: len(x) > 1)
    .sort_values("file_hash")
)

print("=" * 70)
print("EXACT DUPLICATE AUDIT")
print("=" * 70)

print(
    "Number of duplicate files:",
    len(duplicate_groups)
)

print(
    "Number of duplicate groups:",
    duplicate_groups["file_hash"].nunique()
)

if len(duplicate_groups) > 0:

    display(duplicate_groups)

else:

    print("PASS — No exact duplicate files detected.")

EXACT DUPLICATE AUDIT
Number of duplicate files: 0
Number of duplicate groups: 0
PASS — No exact duplicate files detected.


In [8]:
# ============================================================
# CROSS-SPLIT EXACT DUPLICATE CHECK
# ============================================================

hash_split_table = (
    hash_df
    .groupby("file_hash")["split"]
    .agg(lambda x: sorted(set(x)))
)

cross_split_duplicates = hash_split_table[
    hash_split_table.apply(lambda x: len(x) > 1)
]

print("=" * 70)
print("CROSS-SPLIT EXACT DUPLICATE AUDIT")
print("=" * 70)

print(
    "Cross-split duplicate groups:",
    len(cross_split_duplicates)
)

if len(cross_split_duplicates) == 0:

    print("PASS — No exact image leakage between train/val/test.")

else:

    print("WARNING — Potential data leakage detected.")

    for file_hash, splits in cross_split_duplicates.items():

        print(
            file_hash,
            "appears in:",
            splits
        )

CROSS-SPLIT EXACT DUPLICATE AUDIT
Cross-split duplicate groups: 0
PASS — No exact image leakage between train/val/test.


In [9]:
# ============================================================
# SAVE AUDIT DATA
# ============================================================

audit_output = "/content/central_asian_food_audit.csv"

dimensions_df.to_csv(
    audit_output,
    index=False
)

print("Saved audit file:")
print(audit_output)

Saved audit file:
/content/central_asian_food_audit.csv


In [10]:
# ============================================================
# CONFIGURATION
# ============================================================

import os
import random
import numpy as np
import torch

# Dataset
DATASET_ROOT = path          # kagglehub dataset path

# Model
MODEL_NAME = "efficientnet_b0"
NUM_CLASSES = 42
IMAGE_SIZE = 224

# Training
BATCH_SIZE = 32
EPOCHS_STAGE1 = 5
EPOCHS_STAGE2 = 15

LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cuda


In [ ]:
# ============================================================
# REPRODUCIBILITY + OUTPUT CONFIGURATION
# ============================================================

SEED = 42

IMAGE_SIZE = 224
BATCH_SIZE = 32
# NUM_CLASSES is set in the config cell and recomputed from the dataset in the
# class-mapping cell. Reading class_names here would run before it is defined.
MODEL_NAME = "efficientnet_b0"

EPOCHS_STAGE1 = 5
EPOCHS_STAGE2 = 15

LR_STAGE1 = 1e-3
LR_STAGE2 = 1e-4
WEIGHT_DECAY = 1e-4

NUM_WORKERS = 2

OUTPUT_DIR = Path("/content/central_asian_food_model_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_STAGE1_PATH = OUTPUT_DIR / "best_baseline_stage1.pth"
BEST_FINETUNED_PATH = OUTPUT_DIR / "best_efficientnet_b0_finetuned.pth"
PRODUCTION_MODEL_PATH = OUTPUT_DIR / "efficientnet_b0_central_asian_food_v1.pth"
CLASS_MAPPING_PATH = OUTPUT_DIR / "class_mapping.json"
CONFIG_PATH = OUTPUT_DIR / "model_config.json"
METRICS_PATH = OUTPUT_DIR / "metrics.json"
REPORT_PATH = OUTPUT_DIR / "classification_report.csv"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Seed:", SEED)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)

In [ ]:
import cv2
import pandas as pd
from pathlib import Path

import albumentations as A

from albumentations.pytorch import ToTensorV2

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights

import torch.nn as nn

# Required by the evaluation, plotting and export cells below.
import json
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)


In [13]:
train_transform = A.Compose([

    A.LongestMaxSize(max_size=256),

    A.PadIfNeeded(
        min_height=256,
        min_width=256,
        border_mode=cv2.BORDER_CONSTANT
    ),

    A.RandomCrop(224,224),

    A.HorizontalFlip(p=0.5),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=15,
        p=0.5
    ),

    A.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
        p=0.5
    ),

    A.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    ),

    ToTensorV2()

])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [14]:
val_transform = A.Compose([

    A.LongestMaxSize(max_size=256),

    A.PadIfNeeded(
        min_height=256,
        min_width=256,
        border_mode=cv2.BORDER_CONSTANT
    ),

    A.CenterCrop(224,224),

    A.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    ),

    ToTensorV2()

])

In [15]:
A.Resize(224,224)

Resize(p=1.0, area_for_downscale=None, height=224, interpolation=1, mask_interpolation=0, width=224)

In [16]:
class CentralAsianFoodDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = cv2.imread(row.path)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label = int(row.label)

        if self.transform:

            image = self.transform(image=image)["image"]

        return image, label

In [ ]:
# ============================================================
# BUILD ONE GLOBAL CLASS MAPPING
# ============================================================
# IMPORTANT:
# The model output index must mean the same class in train/val/test.
# We therefore create the mapping ONCE from the training classes and
# apply that mapping to every split.

train_split_path = Path(DATASET_ROOT) / "train"

class_names = sorted([
    p.name for p in train_split_path.iterdir()
    if p.is_dir()
])

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(class_names)
}

NUM_CLASSES = len(class_names)

records = []

for split in ["train", "val", "test"]:

    split_path = Path(DATASET_ROOT) / split

    split_classes = sorted([
        p.name for p in split_path.iterdir()
        if p.is_dir()
    ])

    # Verify every split contains exactly the same classes.
    if set(split_classes) != set(class_names):
        raise ValueError(
            f"Class mismatch in {split}. "
            f"Expected {len(class_names)} classes, found {len(split_classes)}."
        )

    for cls in split_classes:

        cls_path = split_path / cls

        for img in cls_path.rglob("*"):

            if img.is_file() and img.suffix.lower() in IMAGE_EXTENSIONS:

                records.append({
                    "split": split,
                    "class": cls,
                    "label": class_to_idx[cls],
                    "path": str(img),
                    "filename": img.name,
                })

df = pd.DataFrame(records)

print("Number of classes:", NUM_CLASSES)
print("Total images:", len(df))
print()
for i, name in enumerate(class_names):
    print(f"{i:02d} -> {name}")

In [18]:
train_df = df[df.split=="train"]

val_df = df[df.split=="val"]

test_df = df[df.split=="test"]

print(len(train_df))

print(len(val_df))

print(len(test_df))

10969
2735
2698


In [19]:
train_dataset = CentralAsianFoodDataset(
    train_df,
    train_transform
)

val_dataset = CentralAsianFoodDataset(
    val_df,
    val_transform
)

test_dataset = CentralAsianFoodDataset(
    test_df,
    val_transform
)

In [20]:
train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=True

)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True

)

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True

)

In [21]:
weights = EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(weights=weights)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 149MB/s]


In [22]:
for param in model.features.parameters():

    param.requires_grad=False

In [23]:
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(

    nn.Dropout(0.3),

    nn.Linear(
        in_features,
        NUM_CLASSES
    )

)

model=model.to(DEVICE)

In [24]:
criterion = nn.CrossEntropyLoss()

In [25]:
optimizer = torch.optim.AdamW(

    model.classifier.parameters(),

    lr=LR_STAGE1,

    weight_decay=WEIGHT_DECAY

)

In [26]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=EPOCHS_STAGE1

)

In [27]:
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_2161/2340218076.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [28]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
):

    model.train()

    running_loss = 0

    predictions = []

    labels_list = []

    for images, labels in loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        preds = outputs.argmax(1)

        predictions.extend(preds.cpu().numpy())

        labels_list.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader)

    epoch_acc = accuracy_score(labels_list, predictions)

    epoch_f1 = f1_score(
        labels_list,
        predictions,
        average="macro"
    )

    return epoch_loss, epoch_acc, epoch_f1

In [29]:
def validate(
    model,
    loader,
    criterion,
    device,
):

    model.eval()

    running_loss = 0

    predictions = []

    labels_list = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            preds = outputs.argmax(1)

            predictions.extend(preds.cpu().numpy())

            labels_list.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader)

    epoch_acc = accuracy_score(labels_list, predictions)

    epoch_f1 = f1_score(
        labels_list,
        predictions,
        average="macro"
    )

    return epoch_loss, epoch_acc, epoch_f1

In [30]:
best_f1 = 0

patience = 5

counter = 0

In [ ]:
# REMOVED: this cell saved model.state_dict() BEFORE any training had run,
# producing a checkpoint whose classifier head was still at random
# initialization. Checkpoints are written by the training loops only.


In [ ]:
best_f1 = 0.0
patience = 5
counter = 0

for epoch in range(EPOCHS_STAGE1):

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE,
    )

    val_loss, val_acc, val_f1 = validate(
        model,
        val_loader,
        criterion,
        DEVICE,
    )

    scheduler.step()

    print(
        f"Epoch {epoch+1}/{EPOCHS_STAGE1}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Acc : {train_acc:.4f}"
    )

    print(
        f"Train F1  : {train_f1:.4f}"
    )

    print(
        f"Val Loss  : {val_loss:.4f}"
    )

    print(
        f"Val Acc   : {val_acc:.4f}"
    )

    print(
        f"Val F1    : {val_f1:.4f}"
    )

    if val_f1 > best_f1:

        best_f1 = val_f1

        counter = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "stage": "stage1_head_only",
                "best_val_f1": best_f1,
                "best_epoch": epoch + 1,
            },
            BEST_STAGE1_PATH,
        )

    else:

        counter += 1

    if counter >= patience:

        print("Early stopping.")

        break

# ============================================================
# STAGE 2 — FINE-TUNE EFFICIENTNET-B0
# ============================================================

The baseline freezes the ImageNet feature extractor.  
Now we adapt the final three EfficientNet feature blocks to the
Central Asian food domain.

Model selection is based ONLY on validation Macro F1.

The test set remains untouched until the final evaluation.

In [ ]:
# ============================================================
# LOAD BEST STAGE 1 CHECKPOINT
# ============================================================

stage1_checkpoint = torch.load(
    BEST_STAGE1_PATH,
    map_location=DEVICE,
)

model.load_state_dict(stage1_checkpoint["model_state_dict"])

# Freeze all feature layers first.
for param in model.features.parameters():
    param.requires_grad = False

# Fine-tune the final three feature blocks.
for param in model.features[-3:].parameters():
    param.requires_grad = True

# The classifier is always trainable.
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters:     {total:,}")
print(f"Trainable ratio:      {trainable / total:.2%}")

In [ ]:
# ============================================================
# STAGE 2 OPTIMIZER + SCHEDULER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_STAGE2,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS_STAGE2,
)

In [ ]:
# ============================================================
# STAGE 2 TRAINING LOOP
# ============================================================
# Critical order inside every training batch:
#
#   optimizer.zero_grad()
#   forward pass
#   loss
#   loss.backward()
#   optimizer.step()
#
# The scheduler is stepped ONCE after the complete epoch.
# This avoids the scheduler warning encountered in the first attempt.

best_val_f1 = -1.0
best_epoch = -1
history = []

for epoch in range(EPOCHS_STAGE2):

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE,
    )

    val_loss, val_acc, val_f1 = validate(
        model,
        val_loader,
        criterion,
        DEVICE,
    )

    # Correct scheduler order:
    # optimizer.step() already happened inside train_one_epoch().
    scheduler.step()

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
    })

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS_STAGE2} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # Save the best checkpoint according to validation Macro F1.
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": MODEL_NAME,
                "num_classes": NUM_CLASSES,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "image_size": IMAGE_SIZE,
                "normalization": {
                    "mean": [0.485, 0.456, 0.406],
                    "std": [0.229, 0.224, 0.225],
                },
                "best_val_f1": best_val_f1,
                "best_epoch": best_epoch,
                "stage": "fine_tuned",
            },
            BEST_FINETUNED_PATH,
        )

        print("✓ Best checkpoint saved")

history_df = pd.DataFrame(history)

print()
print("Best validation Macro F1:", f"{best_val_f1:.4f}")
print("Best epoch:", best_epoch)

# ============================================================
# TRAINING CURVES
# ============================================================

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    history_df["epoch"],
    history_df["train_acc"],
    label="Train Accuracy",
)

ax.plot(
    history_df["epoch"],
    history_df["val_acc"],
    label="Validation Accuracy",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Fine-Tuning Accuracy")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    history_df["epoch"],
    history_df["train_f1"],
    label="Train Macro F1",
)

ax.plot(
    history_df["epoch"],
    history_df["val_f1"],
    label="Validation Macro F1",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Macro F1")
ax.set_title("Fine-Tuning Macro F1")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ============================================================
# LOAD THE BEST CHECKPOINT
# ============================================================

In [ ]:
checkpoint = torch.load(
    BEST_FINETUNED_PATH,
    map_location=DEVICE,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(DEVICE)

print("Loaded best checkpoint")
print("Best validation Macro F1:", checkpoint["best_val_f1"])
print("Best epoch:", checkpoint["best_epoch"])

# ============================================================
# FINAL TEST EVALUATION
# ============================================================

This is the final held-out evaluation.  
Do not use these results to tune the model afterward.

In [ ]:
@torch.no_grad()
def evaluate_topk(model, loader, device, k=3):

    model.eval()

    total = 0
    top1_correct = 0
    topk_correct = 0

    all_labels = []
    all_preds = []
    all_probabilities = []

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        _, topk_preds = probabilities.topk(
            k,
            dim=1
        )

        total += labels.size(0)

        top1_correct += (
            topk_preds[:, 0] == labels
        ).sum().item()

        topk_correct += (
            topk_preds == labels.unsqueeze(1)
        ).any(dim=1).sum().item()

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_preds.extend(
            topk_preds[:, 0].cpu().numpy()
        )

        all_probabilities.append(
            probabilities.cpu()
        )

    all_probabilities = torch.cat(
        all_probabilities
    ).numpy()

    return {
        "top1": top1_correct / total,
        "topk": topk_correct / total,
        "y_true": np.array(all_labels),
        "y_pred": np.array(all_preds),
        "probabilities": all_probabilities,
    }


test_results = evaluate_topk(
    model,
    test_loader,
    DEVICE,
    k=3,
)

test_top1 = test_results["top1"]
test_top3 = test_results["topk"]

y_true = test_results["y_true"]
y_pred = test_results["y_pred"]

test_probabilities = test_results["probabilities"]

test_macro_precision = precision_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

test_macro_recall = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

test_macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0,
)

test_weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0,
)

print("=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)
print(f"Top-1 Accuracy : {test_top1:.4%}")
print(f"Top-3 Accuracy : {test_top3:.4%}")
print(f"Macro Precision: {test_macro_precision:.4%}")
print(f"Macro Recall   : {test_macro_recall:.4%}")
print(f"Macro F1       : {test_macro_f1:.4%}")
print(f"Weighted F1    : {test_weighted_f1:.4%}")

# ============================================================
# PER-CLASS PERFORMANCE
# ============================================================

In [ ]:
report_dict = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).T

display(report_df)

report_df.to_csv(
    REPORT_PATH
)

print("Saved:", REPORT_PATH)

# ============================================================
# CONFUSION MATRIX
# ============================================================

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
)

fig, ax = plt.subplots(
    figsize=(18, 16)
)

im = ax.imshow(cm)

ax.set_xticks(
    range(NUM_CLASSES)
)

ax.set_xticklabels(
    class_names,
    rotation=90
)

ax.set_yticks(
    range(NUM_CLASSES)
)

ax.set_yticklabels(
    class_names
)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

ax.set_title(
    "EfficientNet-B0 — Final Test Confusion Matrix"
)

fig.colorbar(
    im,
    ax=ax
)

plt.tight_layout()
plt.show()

# ============================================================
# DIFFICULT CLASS ANALYSIS
# ============================================================

In [ ]:
per_class = report_df.loc[
    class_names,
    [
        "precision",
        "recall",
        "f1-score",
        "support",
    ],
]

print("Weakest classes:")
display(
    per_class.sort_values(
        "f1-score"
    ).head(10)
)

print("Best classes:")
display(
    per_class.sort_values(
        "f1-score",
        ascending=False
    ).head(10)
)

In [ ]:
# Example: inspect the remaining difficult food families.
difficult_classes = [
    "asip",
    "shashlyk-chicken",
    "shashlyk-chicken-v",
    "shashlyk-kuskovoi",
    "shashlyk-kuskovoi-v",
    "shashlyk-minced-meat",
    "beshbarmak-w-kazy",
    "beshbarmak-wo-kazy",
]

available = [
    c for c in difficult_classes
    if c in per_class.index
]

display(
    per_class.loc[available]
    .sort_values("f1-score")
)

# ============================================================
# SINGLE-IMAGE PRODUCTION INFERENCE
# ============================================================

In [ ]:
IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406,
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225,
)

def preprocess_image(image_path):

    # Read image using OpenCV.
    image = cv2.imread(
        str(image_path)
    )

    if image is None:
        raise FileNotFoundError(
            f"Could not read image: {image_path}"
        )

    # OpenCV loads BGR; Albumentations/model expects RGB.
    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    transformed = val_transform(
        image=image
    )["image"]

    return transformed.unsqueeze(0)


@torch.no_grad()
def predict_image(
    image_path,
    model=model,
    top_k=3,
    device=DEVICE,
):

    model.eval()

    image_tensor = preprocess_image(
        image_path
    ).to(device)

    logits = model(
        image_tensor
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    values, indices = probabilities.topk(
        top_k,
        dim=1
    )

    values = values[0].cpu().numpy()
    indices = indices[0].cpu().numpy()

    predictions = []

    for confidence, index in zip(
        values,
        indices
    ):

        predictions.append({
            "class": class_names[int(index)],
            "class_index": int(index),
            "confidence": float(confidence),
        })

    return {
        "top_prediction": predictions[0],
        "top_k": predictions,
    }

In [ ]:
# ============================================================
# OPTIONAL: TEST ONE REAL IMAGE
# ============================================================

IMAGE_TO_TEST = None

# Example:
# IMAGE_TO_TEST = "/content/my_food_photo.jpg"

if IMAGE_TO_TEST is None:

    print(
        "Set IMAGE_TO_TEST to a real image path "
        "to test the production inference function."
    )

else:

    result = predict_image(
        IMAGE_TO_TEST,
        top_k=3,
    )

    print(
        json.dumps(
            result,
            indent=2,
            ensure_ascii=False,
        )
    )

# ============================================================
# SAVE FINAL PRODUCTION ARTIFACTS
# ============================================================

In [ ]:
# Store all information required by the inference application.
production_checkpoint = {

    "model_state_dict":
        model.state_dict(),

    "model_name":
        MODEL_NAME,

    "num_classes":
        NUM_CLASSES,

    "class_names":
        class_names,

    "class_to_idx":
        class_to_idx,

    "image_size":
        IMAGE_SIZE,

    "normalization": {
        "mean":
            list(IMAGENET_MEAN),

        "std":
            list(IMAGENET_STD),
    },

    "best_val_f1":
        float(
            checkpoint["best_val_f1"]
        ),

    "best_epoch":
        int(
            checkpoint["best_epoch"]
        ),

    "test_metrics": {

        "top1_accuracy":
            float(test_top1),

        "top3_accuracy":
            float(test_top3),

        "macro_precision":
            float(test_macro_precision),

        "macro_recall":
            float(test_macro_recall),

        "macro_f1":
            float(test_macro_f1),

        "weighted_f1":
            float(test_weighted_f1),
    },

    "version":
        "v1.0",
}

torch.save(
    production_checkpoint,
    PRODUCTION_MODEL_PATH
)

# Save class mapping separately.
with open(
    CLASS_MAPPING_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "class_to_idx":
                class_to_idx,

            "idx_to_class":
                {
                    str(i): name
                    for i, name
                    in enumerate(class_names)
                },
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

# Save preprocessing/model configuration.
with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "model_name":
                MODEL_NAME,

            "num_classes":
                NUM_CLASSES,

            "image_size":
                IMAGE_SIZE,

            "mean":
                list(IMAGENET_MEAN),

            "std":
                list(IMAGENET_STD),

            "version":
                "v1.0",
        },
        f,
        indent=2,
    )

# Save metrics.
with open(
    METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        {
            "top1_accuracy":
                float(test_top1),

            "top3_accuracy":
                float(test_top3),

            "macro_precision":
                float(test_macro_precision),

            "macro_recall":
                float(test_macro_recall),

            "macro_f1":
                float(test_macro_f1),

            "weighted_f1":
                float(test_weighted_f1),

            "best_validation_f1":
                float(checkpoint["best_val_f1"]),

            "best_epoch":
                int(checkpoint["best_epoch"]),

            "version":
                "v1.0",
        },
        f,
        indent=2,
    )

print("Production model:")
print(PRODUCTION_MODEL_PATH)
print()
print("Class mapping:")
print(CLASS_MAPPING_PATH)
print()
print("Model config:")
print(CONFIG_PATH)
print()
print("Metrics:")
print(METRICS_PATH)

# ============================================================
# FINAL ARTIFACT CHECK
# ============================================================

In [ ]:
artifacts = [
    BEST_FINETUNED_PATH,
    PRODUCTION_MODEL_PATH,
    CLASS_MAPPING_PATH,
    CONFIG_PATH,
    METRICS_PATH,
    REPORT_PATH,
]

artifact_rows = []

for artifact in artifacts:

    artifact = Path(artifact)

    artifact_rows.append({
        "artifact": str(artifact),
        "exists": artifact.exists(),
        "size_mb": (
            round(
                artifact.stat().st_size
                / (1024 * 1024),
                2,
            )
            if artifact.exists()
            else None
        ),
    })

display(
    pd.DataFrame(
        artifact_rows
    )
)

# ============================================================
# FINAL SUMMARY
# ============================================================

The classifier is now ready to be used as the image-recognition layer.

Expected artifact structure:

```text
central_asian_food_model_v1/
├── best_efficientnet_b0_finetuned.pth
├── efficientnet_b0_central_asian_food_v1.pth
├── class_mapping.json
├── model_config.json
├── metrics.json
└── classification_report.csv
```

The next application layer is separate:

```text
Image
  ↓
EfficientNet-B0
  ↓
food class / Top-3 / confidence
  ↓
Nutrition database
  ↓
portion calculation
  ↓
calories / protein / carbs / fat / fiber
  ↓
FastAPI
  ↓
Streamlit
```

**Do not put nutrition values inside the neural-network checkpoint.**
Keep the classifier and nutrition database independent so nutrition
data can be corrected or updated without retraining the image model.